# ml Quickstart

**ml** is a unified tabular ML package. One import, the full [Hastie workflow](https://hastie.su.domains/ElemStatLearn/) in code.

This notebook shows the complete workflow from raw data to a validated, production-ready model. Every step is one function call.

```
pip install mlw          # core
pip install mlw[xgboost] # + XGBoost (recommended)
```

In [ ]:
import ml
print(f"ml version: {ml.__version__}")

## 1. Load Data

ml ships two datasets for offline use (`tips`, `flights`). Most others download from OpenML on first use.

In [ ]:
# tips ships bundled — no internet needed
data = ml.dataset("tips")
data.head(3)

In [ ]:
# Profile the data — shape, types, missing, target distribution
prof = ml.profile(data, "tip")
prof

## 2. Split

Three-way split: train (60%) → valid (20%) → test (20%). The `.dev` property gives you train+valid combined for final retraining.

Always use `seed=`. This is enforced.

In [ ]:
s = ml.split(data, "tip", seed=42)
print(f"train: {len(s.train)} | valid: {len(s.valid)} | test: {len(s.test)} | dev: {len(s.dev)}")

## 3. Screen

Run all algorithms quickly to find the best starting point. No tuning yet.

In [ ]:
leaderboard = ml.screen(s, "tip", seed=42)
leaderboard

## 4. Fit

Train a model. cv=5 by default (cross-validation on the training set).

In [ ]:
model = ml.fit(s.train, "tip", algorithm="random_forest", seed=42)
model

## 5. Evaluate

`evaluate()` is the **practice exam** — use it freely on validation data. Iterate until satisfied.

In [ ]:
metrics = ml.evaluate(model, s.valid)
metrics

## 6. Explain

Feature importance — which inputs drive predictions most?

In [ ]:
imp = ml.explain(model)
imp

## 7. Tune (optional)

Hyperparameter search. `n_trials=20` is fast enough for a quickstart.

In [ ]:
tuned = ml.tune(s.train, "tip", algorithm="random_forest", seed=42, n_trials=20)
ml.evaluate(tuned, s.valid)

## 8. Compare

Side-by-side comparison of pre-fitted models. No refitting — fair evaluation.

In [ ]:
lb = ml.compare([model, tuned], s.valid)
lb

In [ ]:
print(f"Best algorithm: {lb.best}")

## 9. Finalize

Retrain on all dev data (train + valid). Same algorithm, bigger dataset.

In [ ]:
final = ml.fit(s.dev, "tip", algorithm="random_forest", seed=42)
print(f"Final model trained on {final.n_train} rows")

## 10. Validate

Optional pass/fail gate before deployment. Define your own rules.

In [ ]:
gate = ml.validate(final, test=s.test, rules={"rmse": "<1.0"})
gate

## 11. Assess

`assess()` is the **final exam** — do this ONCE on held-out test data. ml tracks usage and warns if you repeat it.

In [ ]:
verdict = ml.assess(final, test=s.test)
verdict

## 12. Save & Load

In [ ]:
ml.save(final, "tip_model.mlw")
loaded = ml.load("tip_model.mlw")
predictions = ml.predict(loaded, s.test)
predictions.head()

---

## The full workflow in 12 lines

```python
import ml
data = ml.dataset("tips")
s    = ml.split(data, "tip", seed=42)
leaderboard = ml.screen(s, "tip", seed=42)
model  = ml.fit(s.train, "tip", algorithm="random_forest", seed=42)
tuned  = ml.tune(s.train, "tip", algorithm="random_forest", seed=42)
ml.evaluate(tuned, s.valid)
final  = ml.fit(s.dev, "tip", algorithm="random_forest", seed=42)
gate   = ml.validate(final, test=s.test, rules={"rmse": "<1.0"})
verdict = ml.assess(final, test=s.test)
ml.save(final, "model.mlw")
```

**Next:** See [notebooks/02_explore.ipynb](02_explore.ipynb) for data profiling and feature analysis, or [03_real_world.ipynb](03_real_world.ipynb) for a real dataset example.